<a href="https://colab.research.google.com/github/PacktPublishing/Modern-Computer-Vision-with-PyTorch-2E/blob/main/Chapter16/Unet_Components_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
try:
  from torch_snippets import *
except:
  %pip install -U diffusers torch_snippets lovely_tensors torchinfo

Summary of unet model used in "Implementing Diffusion from Scratch" section

In [2]:
from diffusers import UNet2DModel
from torchinfo import summary

net = UNet2DModel(
  sample_size=28, # the target image resolution
  in_channels=1, # the number of input channels, 3 for RGB images
  out_channels=1, # the number of output channels
  layers_per_block=1, # how many ResNet layers to use per UNet block block_out_channels=(32, 64, 128, 256), # Roughly matching our basic
  down_block_types=(
    "DownBlock2D", # a regular ResNet downsampling block "AttnDownBlock2D", # a ResNet downsampling block with spatial self-attention
    "AttnDownBlock2D",
    "AttnDownBlock2D",
    "AttnDownBlock2D",
  ),
  up_block_types=(
    "AttnUpBlock2D",
    "AttnUpBlock2D",
    "AttnUpBlock2D", # a ResNet upsampling block with spatial self- attention
    "UpBlock2D", # a regular ResNet upsampling block
  )
)

summary(net, depth=2)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Layer (type:depth-idx)                        Param #
UNet2DModel                                   --
├─Conv2d: 1-1                                 2,240
├─Timesteps: 1-2                              --
├─TimestepEmbedding: 1-3                      --
│    └─Linear: 2-1                            201,600
│    └─SiLU: 2-2                              --
│    └─Linear: 2-3                            803,712
├─ModuleList: 1-4                             --
│    └─DownBlock2D: 2-4                       1,557,248
│    └─AttnDownBlock2D: 2-5                   5,826,688
│    └─AttnDownBlock2D: 2-6                   13,557,152
│    └─AttnDownBlock2D: 2-7                   17,272,640
├─ModuleList: 1-5                             --
│    └─AttnUpBlock2D: 2-8                     59,838,912
│    └─AttnUpBlock2D: 2-9                     35,095,200
│    └─AttnUpBlock2D: 2-10                    15,870,400
│    └─UpBlock2D: 2-11                        3,818,304
├─UNetMidBlock2D: 1-6                  

Summary of unet model used in "Understanding Stable Diffusion" section, i.e., summary of pretrained model

In [4]:
from diffusers import StableDiffusionPipeline
model_id = "stabilityai/stable-diffusion-2-1-base"
pipe = StableDiffusionPipeline.from_pretrained(model_id)
pipe = pipe.to('cuda')

Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6a31a2e1-485ba34c4ec9dca72c58c871;5a83d23f-1599-4a11-94a3-3db0c9a53b90)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1-base.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password..
Will try to load from local cache.


OSError: Cannot load model stabilityai/stable-diffusion-2-1-base: model is not cached locally and an error occurred while trying to fetch metadata from the Hub. Please check out the root cause in the stacktrace above.

In [ ]:
from torch_snippets import *
from torch_snippets.trainer.hooks import print_module_ios_for

keep = {'CrossAttnDownBlock2D', 'CrossAttnUpBlock2D', 'UNetMidBlock2DCrossAttn', 'UpBlock2D', 'DownBlock2D'}

with print_module_ios_for(pipe.unet, print_only=keep, required_children='conv_in'):
  image = pipe("a dog sitting on the grass", num_inference_steps=1).images[0]


In [ ]:
summary(pipe.unet, depth=2)